# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guided example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

All references to record sets, fields, and columns use their unique `@id` as defined in the schema.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using `mlcroissant`. The loaded metadata provides overview info and allows access to the structure for exploration.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata object (not dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and column `@id`s.

We'll iterate through record sets, printing each one's `@id`, its fields' `@id`s and descriptions, and sample some records.


In [ ]:
# List record sets from the dataset metadata
record_sets = metadata.recordSet
if len(record_sets) == 0:
    print("No record sets found in metadata. Please verify schema structure.")
else:
    print(f"Found {len(record_sets)} record set(s):")
    for rset in record_sets:
        print(f"- Record Set @id: {rset['@id']} / type: {rset.get('@type')}")
        fields = rset['field'] if 'field' in rset else []
        for field in fields:
            print(f"  - Field @id: {field['@id']} | Name: {field.get('name')} | DataType: {field.get('dataType')}")
        # Print a sample record
        sample_records = list(dataset.records(record_set=rset['@id']))[:1]
        if sample_records:
            print(f"  Sample record (first row):\n   {sample_records[0]}")

## 3. Data Extraction
Load data from available record sets into pandas DataFrames using the `@id` for each record set.

This step enables easy analysis and manipulation of records. We'll print all available record set `@id`s and load each one.

In [ ]:
# Collect all record set @ids
record_set_ids = [rset['@id'] for rset in metadata.recordSet]
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded {len(df)} records from record set: {rset_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head(3))
    else:
        print(f"No records loaded for record set: {rset_id}")
# Choose the first populated record set for further analysis
main_record_set_id = next(iter(dataframes.keys())) if len(dataframes) else None

## 4. Exploratory Data Analysis (EDA)
Demonstrate data transformation and filtering using `@id` references.

- Filter on a numeric field.
- Normalize a numeric field.
- Group data by a categorical field.

In [ ]:
# Select a numeric field (@id) for analysis
if main_record_set_id:
    df = dataframes[main_record_set_id]
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
    else:
        numeric_field_id = df.columns[0]
    print(f"Using numeric field: {numeric_field_id}")

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered rows with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Select a group field (categorical) for grouping
    group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print('No record set DataFrame available for EDA.')

## 5. Visualization

Visualize distribution or relationships between key variables using Matplotlib or Seaborn.

In [ ]:
# Plot the numeric field distribution and grouped means
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    if group_field:
        plt.figure(figsize=(8, 4))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("Visualization unavailable: No numeric field present.")

## 6. Conclusion

This notebook demonstrated:
- Loading and inspecting metadata and records with `mlcroissant`, referencing all entities by their `@id`.
- Visualizing and analyzing the main dataset record set with basic filtering, normalization, and grouping operations.
- Making all references explicit via unique Croissant ontology IDs for robust, reproducible data exploration.

For further research or model development, expand analyses to all record sets, fields, and variables accessible from the FAIR^2 dataset.